In [25]:
import pandas as pd

In [32]:
!pip install openpyxl

Looking in indexes: https://nexus/repository/pypi-proxy/simple


In [35]:
# !pip install openpyxl

In [36]:
df = pd.read_excel('Обращения с января 2025.xlsx')

In [37]:
df.columns = ['number', 'date', 'date_off', 'product', 'type', 'sub-type', 'text']

In [38]:
mask = (df['product'] == 'Дебетовая карта')  & (df.type == 'Обслуживание по карте')

In [39]:
df[mask].groupby(['product', 'type', 'sub-type']).count()

number  \
product         type                  sub-type                                                     
Дебетовая карта Обслуживание по карте Mir Pass                                               149   
                                      Клиент не согласен с удержанием НДФЛ                    11   
                                      Не доволен тарифами / тарифной политикой / усло...    1095   
                                      Не согласен с конвертацией / курсовой разницей           3   
                                      Не согласен с лимитом на снятие/перевод                  1   
                                      Не согласен с начисленными процентами                  611   
                                      Несвоевременное зачисление денежных средств            153   
                                      Несогласие с образованием технической / дебитор...     161   
                                      Отказ/ошибка в проведении операций по карте            147   
                                      Отмена ежегодной комиссии за РКО                       581   
                                      Отмена ежемесячной комиссии за РКО                     684   
                                      Отмена комиссии за неактивный счет                      10   
                                      Отмена комиссии за обслуживание неактивного кар...     325   
                                      Отмена комиссии за снятие/внесение в стороннем АТМ      35   
                                      Отмена разовой комиссии за РКО                          47   
                                      Платеж / перевод не исполнен / исполнен несвоев...     174   
                                      Проблема с 3-D secure / не приходят Смс / Push          30   
                                      Проблема с активацией карты                             11   
                                      Проблема с перевыпуском карты                           41   
                                      Разблокировка средств по операции                       84   

                                                                                          date  \
product         type                  sub-type                                                   
Дебетовая карта Обслуживание по карте Mir Pass                                             149   
                                      Клиент не согласен с удержанием НДФЛ                  11   
                                      Не доволен тарифами / тарифной политикой / усло...  1095   
                                      Не согласен с конвертацией / курсовой разницей         3   
                                      Не согласен с лимитом на снятие/перевод                1   
                                      Не согласен с начисленными процентами                611   
                                      Несвоевременное зачисление денежных средств          153   
                                      Несогласие с образованием технической / дебитор...   161   
                                      Отказ/ошибка в проведении операций по карте          147   
                                      Отмена ежегодной комиссии за РКО                     581   
                                      Отмена ежемесячной комиссии за РКО                   684   
                                      Отмена комиссии за неактивный счет                    10   
                                      Отмена комиссии за обслуживание неактивного кар...   325   
                                      Отмена комиссии за снятие/внесение в стороннем АТМ    35   
                                      Отмена разовой комиссии за РКО                        47   
                                      Платеж / перевод не исполнен / исполнен несвоев...   174   
                                      Проблема с 3-D secure / не приходят Смс / Push        30   
                           

In [40]:
dfc = df[mask].reset_index(drop=True)

In [41]:
!pip install langchain --user
!pip install langchain langchain-openai
# !pip install grandalf

Looking in indexes: https://nexus/repository/pypi-proxy/simple
Looking in indexes: https://nexus/repository/pypi-proxy/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 92.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 240.4 MB/s eta 0:00:00


In [42]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [211]:
LLM_KWARGS = {
    # 'top_p':0.95, 
    # 'top_k':20, 
    'max_tokens':8192*2*2,
    # 'temperature':0.1
}
BASE_URL = 'https://gpt.run.fc.uralsibbank.ru/v1'
API_KEY = 'EMPTY'
MODEL = 'openai/gpt-oss-120b'

In [212]:
llm = ChatOpenAI(model=MODEL, 
                 base_url=BASE_URL,
                 api_key=API_KEY, 
                 model_kwargs=LLM_KWARGS,)
# llm_json = ChatOpenAI(model=MODEL, 
#                  base_url=BASE_URL,
#                  api_key=API_KEY, 
#                  model_kwargs={'response-format': {'type':'json_object'}, 'max_tokens':4096})

/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3517: UserWarning: Parameters {'max_tokens'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [213]:
SYSTEM = '''
**Контекст**  
Для анализа обращения — необходимо отделять саму причину жалобы (то, что спровоцировало её возникновение) от её последствий. Причина — это конкретный факт, ошибочная операция или некорректное действие системы (например, неверный расчёт комиссии, ошибка в тарифах, неправильное информирование и т.п.). Последствия (списание средств, блокировка карты, ухудшение сервиса) являются лишь результатом этой причины и не подлежат классификации. При категоризации обращения мы фиксируем именно ту проблему, из‑за которой клиент получил недовольство, т. е. классифицируем её с точки зрения клиента, а не с точки зрения внутренних процессов банка.
При формировании ответа используйте именно эти названия категорий в поле **issues**.  

**Категории проблем (issues)**  

| **Category** | **Sub‑category** | **Description (кратко)** |
|--------------|------------------|---------------------------|
| **1. Комиссии и сборы** | **1.1 Неожиданная / скрытая комиссия** | Платёж списан без предварительного информирования (комиссия за обслуживание, за снятие, за пополнение и т.п.). |
| **1. Комиссии и сборы** | **1.2 Комиссия за неактивный/неиспользуемый счёт** | Списание за отсутствие активности (дебетовый, бизнес‑счёт, карта «прибыль», премиум‑пакет). |
| **1. Комиссии и сборы** | **1.3 Комиссия за бизнес‑залы / премиум‑услуги** | Плата за доступ к бизнес‑залам, премиум‑картам без согласия. |
| **1. Комиссии и сборы** | **1.4 Комиссия за операции RKO / пополнение** | Списание за расчётно‑кассовое обслуживание, за пополнение через партнёров, за ввод/вывод наличных. |
| **1. Комиссии и сборы** | **1.5 Комиссия за международные/online‑платежи** | Плата за переводы в рамках SBP, SWIFT, онлайн‑покупки, за превышение лимитов. |
| **1. Комиссии и сборы** | **1.6 Неправильный расчёт комиссии** | Ошибочное начисление суммы комиссии (дублирование, слишком высокая ставка). |
| **2. Неавторизованные/незаконные списания** | **2.1 Неавторизованные списания по счету (RUB/EUR/USD)** | Списание средств без согласия владельца (в том числе за страховку, подписки, услуги). |
| **2. Неавторизованные/незаконные списания** | **2.2 Неавторизованные операции в бизнес‑залах** | Неодобренный доступ к бизнес‑залам и последующее списание. |
| **2. Неавторизованные/незаконные списания** | **2.3 Неавторизованное открытие/закрытие счёта** | Открытие нового счета/карты без согласия клиента. |
| **2. Неавторизованные/незаконные списания** | **2.4 Неавторизованные комиссии за услуги** | Плата за услуги, о которых клиент не знал (SMS‑информирование, 3‑DS, кэш‑бэк). |
| **3. Проблемы с картой** | **3.1 Блокировка/заморозка карты** | Карта заблокирована банком без объяснения (по ФЗ‑161, подозрению в fraude и т.п.). |
| **3. Проблемы с картой** | **3.2 Ошибки при выпуске/перевыпуске карты** | Не получена карта, задержка выдачи, неправильный тип карты, карта с ошибочным номером. |
| **3. Проблемы с картой** | **3.3 Проблемы с PIN / 3‑DS / верификацией** | Неправильный номер телефона, отсутствие возможности установить/сменить 3‑DS, ошибка PIN. |
| **3. Проблемы с картой** | **3.4 Неподдерживаемый/некомплатный тип карты** | Карта не подходит для онлайн‑покупок, QR‑платежей, MirAccept, Яндекс Go. |
| **3. Проблемы с картой** | **3.5 Потеря/кража карты и последствия** | Утеря, кража, блокировка, невозможность восстановить средства. |
| **4. Проблемы со счётом и балансом** | **4.1 Неотображение/недоступность средств** | Баланс не показывается, средства «заморожены», отсутствие баланса в приложении. |
| **4. Проблемы со счётом и балансом** | **4.2 Ошибки начисления процентов/пени** | Неправильный расчёт процентов, отсутствие начисления, избыточные пени. |
| **4. Проблемы со счётом и балансом** | **4.3 Неправильное списание/зачисление средств** | Ошибочный перевод, неверный получатель, двойное/тройное списание, возврат на закрытый счёт. |
| **4. Проблемы со счётом и балансом** | **4.4 Долги и задолженности, неясные данные** | Неожиданная задолженность, непонятные обороты, неучтённые операции. |
| **4. Проблемы со счётом и балансом** | **4.5 Закрытие/открытие счёта без уведомления** | Закрытие счета/карты без сообщения, невозможность открыть новый счёт онлайн. |
| **5. Переводы и платежи** | **5.1 Ошибки при переводе (SBP, QR, SWIFT)** | Неправильный получатель, отказ в переводе, задержка зачисления, двойные переводы. |
| **5. Переводы и платежи** | **5.2 Отказы в оплате (POS, онлайн, QR)** | Отклонённые покупки, превышение лимитов, неверный MCC‑код, отказ в сервисе доставки. |
| **5. Переводы и платежи** | **5.3 Необходимость подтверждения/верификации** | Требование кода, 3‑DS, SMS‑подтверждение, невозможность подтвердить операцию. |
| **5. Переводы и платежи** | **5.4 Проблемы с вводом/выводом наличных** | Ошибки при пополнении в банкомате, ограничение приёма банкнот, отсутствие наличных. |
| **6. Премиальные/бонусные программы** | **6.1 Кэш‑бэк и бонусы не начислены** | Пропущенные кэш‑бэки, недополученные бонусы, снижение кэш‑бэка без уведомления. |
| **6. Премиальные/бонусные программы** | **6.2 Ограничения и закрытие премиум‑пакетов** | Отказ в премиум‑обслуживании, закрытие пакета без объяснения, блокировка привилегий. |
| **7. Пенсии и социальные выплаты** | **7.1 Не поступление пенсии/пособий** | Задержка выплаты, отсутствие выплат, неверный номер счёта, потеря выплат. |
| **7. Пенсии и социальные выплаты** | **7.2 Ошибки в расчёте/перечислении социальных средств** | Неверный оборот, отсутствие выплат по соц‑программам. |
| **8. Информационная поддержка и коммуникация** | **8.1 Отсутствие/противоречивая информация** | Нет ответа в чате, разные ответы от колл‑центра и процессинга, неверные консультации. |
| **8. Информационная поддержка и коммуникация** | **8.2 Низкое качество обслуживания** | Непрофессиональное общение, задержки в рассмотрении, отсутствие менеджера. |
| **8. Информационная поддержка и коммуникация** | **8.3 Недостаточная информированность о тарифах/изменениях** | Нет уведомления о новых тарифах, скрытые изменения условий, отсутствие описания комиссии. |
| **8. Информационная поддержка и коммуникация** | **8.4 Нарушение конфиденциальности / утечки данных** | Неавторизованная передача персональных данных, публикация данных клиента. |
| **9. Документы и справки** | **9.1 Отсутствие/запрос справок и выписок** | Не предоставили выписку для суда, нет справки о доходах, отсутствие договоров. |
| **9. Документы и справки** | **9.2 Ошибки в договорах и согласиях** | Нет договора с фондом, отсутствие согласия на списание, нелегальное изменение условий. |
| **10. Технические и системные проблемы** | **10.1 Проблемы мобильного/веб‑приложения** | Ошибки в личном кабинете, отсутствие SMS‑уведомлений, 3‑DS не работает, приложение «зависает». |
| **10. Технические и системные проблемы** | **10.2 Сбои в работе автоплатежей и сервисов** | Не работают автоплатежи, ошибка при оплате картой, сбой QR‑платежа. |
| **10. Технические и системные проблемы** | **10.3 Ограничения доступа к сервисам** | Нет доступа к онлайн‑банкингу, невозможность привязать карту к сервису, отсутствие 3‑DS. |
| **11. Инфраструктурные ограничения** | **11.1 Нет отделения/банкомата в регионе** | Отсутствие ближайшего отделения, партнёрских банкоматов, ограниченный доступ к наличным. |
| **11. Инфраструктурные ограничения** | **11.2 Ограничения лимитов (снятие, пополнение)** | Лимиты в банкоматах, ограничения на пополнение, ограничение на перевод средств. |
| **12. Юридические / нормативные вопросы** | **12.1 Применение 115‑ФЗ, 161‑ФЗ и др.** | Блокировка по законодательным требованиям без объяснения. |
| **12. Юридические / нормативные вопросы** | **12.2 Споры с кредитными бюро (БКИ)** | Ошибки в кредитной истории, необоснованные запросы. |
| **13. Маркетинг и нежелательные контакты** | **13.1 Навязчивая реклама / звонки** | Рекламные звонки, рассылка предложений, невозможность отказаться от новых условий. |
| **14. Прочие/разные** | **14.1 Ошибки реквизитов (БИК, ИНН, номер телефона)** | Неправильный БИК при оплате, неверный ИНН в чеке, устаревший номер телефона. |
| **14. Прочие/разные** | **14.2 Ошибки в классификации MCC‑кода** | Неправильный MCC‑код, приводящий к неверному начислению комиссии. |
| **14. Прочие/разные** | **14.3 Прочие ситуации без отдельной категории** | Утечка данных, нарушение прав потребителей, отсутствие возможности изменить тариф самостоятельно и т.п. |


При классификации учитывай следующее:
1. Если клиент жалуется, что ему **неправильно сообщили о тарифе/комиссии**, выбирай категорию **8. Информационная поддержка и коммуникация** → **8.1 Отсутствие/противоречивая информация** (или 8.3 Недостаточная информированность о тарифах/изменениях, если более точно).

**Запросы клиента (requested_actions)**  
При обработке обращения необходимо выделять каждый конкретный запрос клиента – это чётко сформулированное требование выполнить определённое действие (например, «вернуть комиссию», «открыть новый счёт», «разблокировать карту», «выдать выписку» и т.п.) – и оформлять его в массив requested_actions, где указываются три обязательных поля: category (короткое название действия, используем только указанные в инструкции названия, например «Возврат комиссии», «Разблокировка карты», «Выдача выписки»), sub‑category (при необходимости уточняющая подкатегория) и description (развёрнутое описание запроса с указанием всех деталей, приведённых клиентом). Каждый найденный запрос оформляется отдельным объектом в массиве; если запросов несколько, они перечисляются последовательно. Отклонения от предложенного формата недопустимы – только точные названия категорий и полное описание гарантируют корректную обработку обращения.

При формировании ответа используйте именно эти названия категорий в поле **requested_actions**.  

**Категории запросов клиента  (requested_actions)**  


**Таблица запросов без пустых ячеек (название группы продублировано в каждой строке)**  

| **Category** | **Sub‑category** | **Description (кратко)** |
|-------------------------------|-----------------------------------|----------------------|
| **1. Возврат / возмещение средств** | **1.1 Возврат средств за товар/услугу (билет, страхование, комиссия, штраф)** | Требование вернуть уплаченную сумму полностью или частично. |
| **1. Возврат / возмещение средств** | **1.2 Возврат ошибочно списанных комиссий (EUR, USD, RKO, SMS‑информирование, бизнес‑залы)** | Списание комиссии без согласия/основания – запрос возврата. |
| **1. Возврат / возмещение средств** | **1.3 Возврат переплаты / двойного списания** | Сумма была списана два‑три раза – нужно вернуть излишки. |
| **1. Возврат / возмещение средств** | **1.4 Возврат удержанных средств (пенсия, зарплата, кэшбэк, бонусы)** | Необходимо вернуть удержанные средства, которые должны были быть зачислены. |
| **1. Возврат / возмещение средств** | **1.5 Экспресс‑возврат (через СБП, на карту, наличными)** | Быстрый возврат в случае невозможности завершить перевод. |
| **1. Возврат / возмещение средств** | **1.6 Возврат в виде компенсации (моральный ущерб, за просрочку, за неудобства)** | Компенсация финансовых потерь и/или морального вреда. |
| **2. Корректировка / перерасчёт операций** | **2.1 Перерасчёт процентов / пени** | Пересчёт начисленных процентов, исправление ошибок расчёта. |
| **2. Корректировка / перерасчёт операций** | **2.2 Перерасчёт комиссии / сброс комиссии** | Перерасчёт комиссии, её отмена или уменьшение. |
| **2. Корректировка / перерасчёт операций** | **2.3 Корректировка баланса (дебетовая/кредитная карта)** | Исправление ошибочного остатка, списаний или зачислений. |
| **2. Корректировка / перерасчёт операций** | **2.4 Списание / аннулирование задолженности (долги, пени, овердрафт)** | Полное снятие задолженности или её частичное списание. |
| **2. Корректировка / перерасчёт операций** | **2.5 Корректировка кредитной/дебетовой истории** | Исправление записей о просроченных платежах, штрафах. |
| **2. Корректировка / перерасчёт операций** | **2.6 Перераспределение / переназначение платежей** | Перевод средств между своими счетами/картами. |
| **3. Отмена / прекращение списаний и услуг** | **3.1 Отмена текущей комиссии / автоматических списаний** | Запрет дальнейшего списания комиссии, RKO, подписок и т.п. |
| **3. Отмена / прекращение списаний и услуг** | **3.2 Отмена / аннулирование услуги (премиум‑пакет, овердрафт, страховка)** | Прекращение действия платных сервисов. |
| **3. Отмена / прекращение списаний и услуг** | **3.3 Отмена перевода / транзакции** | Остановить или отменить уже инициированный платёж. |
| **3. Отмена / прекращение списаний и услуг** | **3.4 Прекращение рекламных / сборных звонков** | Блокировать нежелательные телефонные обращения. |
| **4. Управление счётом и картой** | **4.1 Закрытие / анулирование счёта / карты** | Закрыть нежелательный счёт/карту, включая «полное» закрытие. |
| **4. Управление счётом и картой** | **4.2 Перевыпуск / выдача новой карты (бесплатно, с новым сроком)** | Выпуск/замена карты и/или изменение тарифов обслуживания. |
| **4. Управление счётом и картой** | **4.3 Блокировка / разблокировка карты и онлайн‑банка** | Временная блокировка, разблокировка, снятие ограничений. |
| **4. Управление счётом и картой** | **4.4 Снятие ограничений на дистанционные покупки, снятие лимитов** | Увеличение лимитов, снятие блокировок по онлайн‑транзакциям. |
| **4. Управление счётом и картой** | **4.5 Обновление данных (телефон, адрес, 3‑DS, номер телефона)** | Смена контактных данных, изменение настроек 3‑DS. |
| **4. Управление счётом и картой** | **4.6 Привязка / отвязка карты к сервисам (MirAccept, SBP, QR)** | Подключение/отключение карты в внешних приложениях. |
| **5. Информационные запросы и разъяснения** | **5.1 Разъяснение политики комиссии/тарифов/лимитов** | Объяснение, почему начислена комиссия, как формируются тарифы. |
| **5. Информационные запросы и разъяснения** | **5.2 Запрос выписок, расшифровок операций, справок** | Предоставление выписок, детализации движений, справок о доходах. |
| **5. Информационные запросы и разъяснения** | **5.3 Информирование о статусе обращения/запроса** | Уведомление о текущем статусе, результатах расследования. |
| **5. Информационные запросы и разъяснения** | **5.4 Предоставление нормативных актов, договора, условий** | Выдать политику банка, договоры, правила пользования. |
| **5. Информационные запросы и разъяснения** | **5.5 Письменный / официальный ответ (на адрес, в email)** | Составление официального письма‑ответа клиенту. |
| **5. Информационные запросы и разъяснения** | **5.6 Объяснение причин отказа, блокировки, отклонения транзакции** | Детальное обоснование отрицательного решения. |
| **6. Расследование и проверка** | **6.1 Проверка и расследование спорных списаний/транзакций** | Выяснение причины нежелательных списаний, их возврат. |
| **6. Расследование и проверка** | **6.2 Проверка записей звонков, чатов, аудиозаписей** | Анализ клиентского общения для подтверждения фактов. |
| **6. Расследование и проверка** | **6.3 Внутреннее расследование действий сотрудников** | Выяснение возможных нарушений со стороны персонала. |
| **6. Расследование и проверка** | **6.4 Проверка фактов открытия/закрытия счёта, статуса карт** | Подтверждение/отклонение заявлений о создании/закрытии. |
| **6. Расследование и проверка** | **6.5 Проверка расчётов по процентам, кэшбэку, обороту** | Аудит расчётов начислений и проверка корректности. |
| **7. Защита данных и конфиденциальность** | **7.1 Удаление / анонимизация персональных данных (из отзывов, системы)** | Удалить персональные сведения по требованию клиента. |
| **7. Защита данных и конфиденциальность** | **7.2 Прекращение неавторизованного доступа к данным** | Блокировать дальнейшее использование личных данных. |
| **7. Защита данных и конфиденциальность** | **7.3 Обеспечение соответствия обработке персональных данных (GDPR/ФЗ‑152)** | Гарантировать законность обработки клиентской информации. |
| **8. Техническая и сервисная поддержка** | **8.1 Улучшение работы чата / горячей линии, обучение операторов** | Повышение качества общения, предоставление скриптов. |
| **8. Техническая и сервисная поддержка** | **8.2 Настройка / восстановление SMS‑уведомлений, 3‑DS, кода доступа** | Включение/корректировка уведомлений и механизмов верификации. |
| **8. Техническая и сервисная поддержка** | **8.3 Исправление ошибок в мобильном/веб‑приложении (баланс, операции)** | Технические правки в личном кабинете. |
| **8. Техническая и сервисная поддержка** | **8.4 Ускорение обработки заявок, сокращение сроков (экспресс‑расмотрение)** | Приоритетное решение обращений. |
| **8. Техническая и сервисная поддержка** | **8.5 Внедрение новых функций (дистанционная установка кода, автоплатеж)** | Добавление/активация сервисов по запросу клиента. |
| **9. Прочие запросы** | **9.1 Организация встречи / консультации в отделении** | Запрос личного визита сотрудника/встречи. |
| **9. Прочие запросы** | **9.2 Предоставление контактных данных филиала, номера телефона** | Информация о точках обслуживания. |
| **9. Прочие запросы** | **9.3 Перевод средств на другие банковские карты / счета (включая SWIFT)** | Выполнение переводов, в том числе международных. |
| **9. Прочие запросы** | **9.4 Открытие новых счетов / карт (по доверенности, рублевый счёт)** | Создание новых продуктов в рамках текущего профиля. |
| **9. Прочие запросы** | **9.5 Установка банкомата / камеры в офисе** | Техническое обслуживание инфраструктуры. |
| **9. Прочие запросы** | **9.6 Прекращение рекламных рассылок, спама** | Блокировка маркетинговых контактов. |
| **9. Прочие запросы** | **9.7 Согласование условий реструктуризации, рассрочки, каникул по кредиту** | Переговоры о изменении условий кредитования. |
| **9. Прочие запросы** | **9.8 Оформление претензий, жалоб, эскалаций, передача в надзорные органы** | Составление и подача официальных жалоб. |
| **9. Прочие запросы** | **9.9 Предоставление информации о лимите, кэшбэке, бонусах** | Разъяснение правил начисления и использования бонусов. |
| **9. Прочие запросы** | **9.10 Доступ к архивным данным, восстановление закрытых/архивированных счетов** | Выдача исторических выписок, восстановление старых записей. |

### Инструкция для модели

1. **Проанализируйте текст обращения** (в переменной `{{text}}`).  
2. **Определите одну или несколько проблем** и сопоставьте каждую из них с одной из категорий из таблицы выше.  
3. **Сформируйте массив `issues`** – каждый элемент содержит поля `category`, `sub-category` (выбранное из списка категорий и подкатегорий) и `description` (кратко, но полностью отражающее суть проблемы).  
4. **Если в тексте присутствует явный запрос клиента**, сформируйте массив `requested_actions` с соответствующими элементами  – каждый элемент содержит поля `category`, `sub-category` (выбранное из списка категорий и подкатегорий) и `description` (кратко, но полностью отражающее суть запроса).  
5. **Сгенерируйте валидный JSON**:  
   * Корневые ключи – только `issues` и `requested_actions` (в указанном порядке).  
   * Если один из массивов пустой, его можно опустить либо оставить как `[]`.  
   * Не добавляйте никаких дополнительных полей или вложенных структур.  
6. Ни при каких обстоятельствах в ответе  не выводи такие коды (например \u202f, \u00A0, \u200B и др.)! Разряды в числах лучше отделять через пробел, например 20 000 тысяч.


Пример обращения:
В ответе обращения № K-250619-001768 указано, что согласно тарифам банка возвраты тоже учитываются в оборот покупок по карте. Прошу указать в каком пункте тарифов банка указана данная информация. Где указано что повышение процентной ставке  учитывается весь оборот по карте, в условиях я вижу что учитываются расходные операции по карте в размере 20000 руб, а не оборот по карте?',
Пример ответа:
{{'issues': 
    [{{
'category': '8. Информационная поддержка и коммуникация',
   'sub_category': '8.3 Недостаточная информированность о тарифах/изменениях',
   'description': 'Клиент считает, что в тарифах банка нет чёткой информации о том, что возвраты учитываются в оборот покупок и что повышение процентной ставки рассчитывается на весь оборот карты, видит только ограничение в 20000 руб.'
   }}],
 'requested_actions': 
 [{{
     'category': '5. Информационные запросы и разъяснения',
     'sub_category': '5.1 Разъяснение политики комиссии/тарифов/лимитов',
     'description': 'Указать конкретный пункт тарифного документа, где прописано, что возвраты учитываются в оборот покупок по карте и что повышение процентной ставки рассчитывается на весь оборот карты.'
     }}]
}}

На основе вышеизложенных инструкций разметь следующий запрос:

'''

In [214]:
 # и `description` (кратко, но полностью отражающее суть проблемы).   

In [215]:
from typing import List
from pydantic import BaseModel


class Issue(BaseModel):
    category: str          # название категории
    sub_category: str      # подкатегория
    description: str       # короткое пояснение


class RequestedAction(BaseModel):
    category: str
    sub_category: str
    description: str


class Answer(BaseModel):
    issues: List[Issue]                  # список проблем
    requested_actions: List[RequestedAction]   # список требуемых действий

    
llm_json = llm.with_structured_output(Answer)

In [216]:
prompt = ChatPromptTemplate.from_template(f'{SYSTEM} {{text}}')

In [217]:
chain = prompt | llm_json

In [218]:
import tqdm
import numpy  as np
from collections import defaultdict

In [163]:
# Превращаем в массив (по строкам)
arr = dfc.to_numpy()                   # shape = (4000, n_features)

batch_size = 100
n_batches  = int(np.ceil(len(arr) / batch_size))

# Делим массив на n_batches почти одинаковых кусочка
chunks = np.array_split(arr, n_batches)   # возвращает список из n_batches массивов

In [219]:
print('hello')

hello


In [220]:
!mkdir result_new_clf_v1

mkdir: cannot create directory ‘result_new_clf_v1’: File exists


In [221]:
ix = 5

In [222]:
с1 = chain.invoke({'text':dfc.text[ix]})
с1.model_dump()

{'issues': [{'category': '1. Комиссии и сборы',
   'sub_category': '1.3 Комиссия за бизнес‑залы / премиум‑услуги',
   'description': 'Клиент считает удержанную комиссию за проходы в бизнес‑залы неправомерной, ссылаясь на скриншот из приложения Mir Pass, где указано безлимитное предоставление проходов.'},
  {'category': '8. Информационная поддержка и коммуникация',
   'sub_category': '8.1 Отсутствие/противоречивая информация',
   'description': 'Клиент получил противоречивую информацию: тариф банка предполагает комиссию, а приложение Mir Pass обещает безлимитные проходы.'}],
 'requested_actions': [{'category': '6. Расследование и проверка',
   'sub_category': '6.1 Проверка и расследование спорных списаний/транзакций',
   'description': 'Провести проверку удержанной комиссии за бизнес‑залы, сопоставив её с условиями тарифа банка и заявлением о безлимитных проходах в Mir Pass, и определить правомерность списания.'},
  {'category': '5. Информационные запросы и разъяснения',
   'sub_categor

In [224]:
df.shape, dfc.shape

((83139, 7), (4353, 7))

In [225]:
# с1.model_dump()['issues']
# c1.model_dump()['requested_actions']

In [226]:
dfc.columns

Index(['number', 'date', 'date_off', 'product', 'type', 'sub-type', 'text'], dtype='object')

In [227]:
%%time
out = defaultdict(list)
for enum, chunk in enumerate(tqdm.tqdm(chunks[17:]), start=17):
    temp_df = pd.DataFrame(chunk, columns=dfc.columns)
    temp_df = temp_df.iloc[:]
    inputs = [{'text':k} for k in temp_df.text]
    c = chain.batch(inputs, config={'max_concurrency': 5})
    numbers = temp_df.number.tolist()
    date = temp_df.date.tolist()
    
    issues = [c1.model_dump()['issues']  for c1 in c]
    requested_actions = [c1.model_dump()['requested_actions']  for c1 in c]
    
    out['number'].extend(numbers)
    out['date'].extend(date)
    out['issue'].extend(issues)
    out['requested_action'].extend(requested_actions)
    
    a = pd.DataFrame().from_dict(out)
    temp_df = a.copy()
    temp_df.number = temp_df.number.astype(str)
    temp_df.to_parquet(f'./result_new_clf_v1/{enum}.parquet')
    out = defaultdict(list)
    # break
    

100%|██████████| 27/27 [1:07:05<00:00, 149.08s/it]

CPU times: user 13.7 s, sys: 854 ms, total: 14.5 s
Wall time: 1h 7min 5s


In [208]:
temp_df.explode('requested_action')

,number,date,issue,requested_action
0,K-250630-002734,2025-06-30 21:13:23,"[{'category': '1. Комиссии и сборы', 'sub_cate...",{'category': '1. Возврат / возмещение средств'...
1,K-250630-002727,2025-06-30 20:07:44,"[{'category': '1. Комиссии и сборы', 'sub_cate...",{'category': '1. Возврат / возмещение средств'...
2,O-250630-001224,2025-06-30 19:08:41,[{'category': '8. Информационная поддержка и к...,{'category': '1. Возврат / возмещение средств'...


In [201]:
[c1.model_dump()['issues']  for c1 in c]

[[{'category': '1. Комиссии и сборы',
   'sub_category': '1.1 Неожиданная / скрытая комиссия',
   'description': 'Банк списал комиссию за снятие наличных в размере 2093\u202fруб. за 7 операций, хотя клиент не был проинформирован об изменении условий и новых тарифах.'},
  {'category': '8. Информационная поддержка и коммуникация',
   'sub_category': '8.3 Недостаточная информированность о тарифах/изменениях',
   'description': 'Клиент утверждает, что не получил уведомление о повышении порога и введении комиссии 299\u202fруб. за каждое снятие наличных в банкоматах Сбербанка.'}],
 [{'category': '1. Комиссии и сборы',
   'sub_category': '1.1 Неожиданная / скрытая комиссия',
   'description': 'Клиент обнаружил списание 99\u202fрублей с заблокированной карты, не пользовался картой и не согласен с комиссией.'}],
 [{'category': '8. Информационная поддержка и коммуникация',
   'sub_category': '8.3 Недостаточная информированность о тарифах/изменениях',
   'description': 'Клиент не был проинформиро

In [231]:
import os

In [232]:
files = os.listdir('result_old_clf_v2/')

In [234]:
result = pd.concat([pd.read_parquet(f'result_old_clf_v2/{f}')  for f in files], ignore_index=True)

In [235]:
result.explode('answer').shape, result.shape

((4525, 6), (4353, 6))

In [236]:
result2 = result.explode('answer')
result2.answer.value_counts()

Отмена ежегодной комиссии за РКО                                     641
Не согласен с начисленными процентами                                604
Отмена комиссии за снятие/внесение в стороннем АТМ                   511
Отмена ежемесячной комиссии за РКО                                   483
Платеж / перевод не исполнен / исполнен несвоевременно               289
Отмена разовой комиссии за РКО                                       268
Отказ/ошибка в проведении операций по карте                          216
Несогласие с образованием технической / дебиторской задолженности    175
Несвоевременное зачисление денежных средств                          156
Dragon Pass                                                          118
Проблема с перевыпуском карты                                         91
Разблокировка средств по операции                                     80
Не согласен с лимитом на снятие/перевод                               59
Проблема с 3‑D Secure / не приходят SMS / Push     

In [253]:
dfc.number = dfc.number.astype(str)

In [255]:
result3 = dfc[['number', 'date']].merge(result2, left_on=['number'], right_on=['number'])

In [258]:
result3.to_csv('old_res.csv', index=False, sep=';')

In [257]:
result3.explode('answer').to_csv('old_res_v2.csv', index=False, sep=';')

In [35]:
result2 = result.explode('answer')
result2.answer.value_counts()

Отмена комиссии за неактивный счёт                                   848
Не согласен с начисленными процентами                                604
Отмена комиссии за снятие/внесение в стороннем АТМ                   445
Отмена ежегодной комиссии за РКО                                     407
Отмена ежемесячной комиссии за РКО                                   367
Платеж / перевод не исполнен / исполнен несвоевременно               341
Отказ/ошибка в проведении операций по карте                          297
Несогласие с образованием технической / дебиторской задолженности    166
Несвоевременное зачисление денежных средств                          159
Dragon Pass                                                          124
Разблокировка средств по операции                                    101
Отмена разовой комиссии за РКО                                        82
Проблема с перевыпуском карты                                         75
Не согласен с лимитом на снятие/перевод            

In [45]:
# result2[result2.answer=='Dragon Pass'].text.iloc[8]

In [237]:
result2.answer.isna().mean()

0.1518232044198895

In [240]:
(result2['sub-type'] == result2.answer).mean()

0.3323756906077348

In [242]:
result2[(result2['sub-type'] != result2.answer)]

,number,product,type,sub-type,text,answer
4,O-251222-001019,Дебетовая карта,Обслуживание по карте,Не доволен тарифами / тарифной политикой / усл...,ВО ВЛОЖЕНИИ,NaN
7,S-251222-000929,Дебетовая карта,Обслуживание по карте,Отмена ежемесячной комиссии за РКО,1. Прошу вернуть денежные средства списанные з...,Несогласие с образованием технической / дебито...
8,S-251222-000927,Дебетовая карта,Обслуживание по карте,Отмена ежегодной комиссии за РКО,Добрый день! В банке на мое имя открыта дебето...,NaN
10,S-251222-000912,Дебетовая карта,Обслуживание по карте,Не доволен тарифами / тарифной политикой / усл...,"Пользуюсь другим банком, на выгодных условиях",NaN
11,S-251222-000911,Дебетовая карта,Обслуживание по карте,Не доволен тарифами / тарифной политикой / усл...,Пополнял карту через банкомат почта банк. 9 ра...,Отмена комиссии за снятие/внесение в стороннем...
...,...,...,...,...,...,...
4343,K-250811-001078,Дебетовая карта,Обслуживание по карте,Разблокировка средств по операции,Дополнение к прошлой претензии O-250805-000165...,Не согласен с конвертацией / курсовой разницей
4344,K-250811-001075,Дебетовая карта,Обслуживание по карте,Не доволен тарифами / тарифной политикой / усл...,"В связи с тем, что я не был надлежащим образом...",Отмена комиссии за снятие/внесение в стороннем...
4347,K-250809-001018,Дебетовая карта,Обслуживание по карте,Отмена ежемесячной комиссии за РКО,У меня произошло списание за обслуживание карт...,NaN
4348,K-250809-001014,Дебетовая карта,Обслуживание по карте,Не доволен тарифами / тарифной политикой / усл...,Добрый День! Клиенту была произведена блокиров...,Отказ/ошибка в проведении операций по карте
